# AI Policy NLP Analyzer: US vs. Europe
## Comparing Legislative Language, Priorities & Approaches

This notebook applies **Natural Language Processing (NLP)** techniques to compare
how the United States and European Union approach AI legislation.

### Includes:
1. **Build a corpus** of real US and EU AI bills/regulations with full text excerpts
2. **Preprocess text** — tokenize, remove stopwords, normalize
3. **TF-IDF analysis** — find the most statistically distinctive terms per region
4. **Topic modeling (LDA)** — discover latent themes in each region's legislation
5. **Cosine similarity** — measure how similar or different the two corpora are
6. **Keyword co-occurrence** — map which concepts cluster together
7. **Sentiment & tone analysis** — is the language permissive, restrictive, or punitive?
8. **Visual comparison** — charts, heatmaps, and divergence plots

### Requirements
```
pip install scikit-learn numpy pandas scipy matplotlib
```
No internet connection needed — all data is embedded in the corpus below.


## 1. Imports & Setup
We use `scikit-learn` for all NLP heavy lifting, `matplotlib` for charts, and pure Python stdlib for text processing. No NLTK or spaCy required.

In [ ]:
import re
import math
import json
import warnings
import itertools
import collections
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

warnings.filterwarnings('ignore')

# ── Plotting style ──────────────────────────────────────────────────────────
# Clean, publication-quality style with a warm off-white background
plt.rcParams.update({
    'figure.facecolor':  '#f5f2ec',
    'axes.facecolor':    '#f5f2ec',
    'axes.edgecolor':    '#c8c3b8',
    'axes.labelcolor':   '#1a1a18',
    'text.color':        '#1a1a18',
    'xtick.color':       '#5a5a55',
    'ytick.color':       '#5a5a55',
    'axes.grid':         True,
    'grid.color':        '#e0dbd0',
    'grid.linewidth':    0.6,
    'font.family':       'serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

# Color palette: US = warm blue, EU = deep green
US_COLOR  = '#1a3a6a'   # deep navy
EU_COLOR  = '#1a4a2a'   # forest green
US_LIGHT  = '#a8c4e8'
EU_LIGHT  = '#a8d4b8'
GOLD      = '#b8860b'

print("✓ All imports loaded successfully.")
print(f"  scikit-learn, numpy {np.__version__}, pandas {pd.__version__}, matplotlib ready.")


## 2. The Legislative Corpus

We build a structured corpus of **US and EU AI legislation** from 2020–2026.
Each document contains the legislative text, key metadata, and thematic tags.

**US bills included:**
- National AI Initiative Act (2020)
- Executive Order 14110 — Safe, Secure AI (Biden, 2023)
- TAKE IT DOWN Act (2025)
- EO 14318 — Data Center Permitting (2025)
- America's AI Action Plan (2025)
- EO 14365 — National AI Policy Framework (2025)
- SPEED Act — NEPA permitting reform (2025)
- AI Action Plan — National Framework (2026)

**EU instruments included:**
- EU AI Act (2024) — the world's first comprehensive AI law
- GDPR as applied to AI systems (2018/ongoing)
- EU AI Liability Directive (proposed 2022)
- EU Artificial Intelligence Act — High-Risk provisions
- EU AI Office establishment (2024)
- EU AI Act — Prohibited Systems provisions
- Digital Services Act AI provisions (2022)
- EU AI Act — Foundation Model / GPAI provisions (2024)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CORPUS DEFINITION
# Each entry is a legislative document with:
#   id       — short name for display
#   region   — "US" or "EU"
#   date     — YYYY-MM for sorting
#   status   — legislative status
#   focus    — primary thematic area(s)
#   text     — the full text excerpt used for NLP (longer = better analysis)
# ─────────────────────────────────────────────────────────────────────────────

CORPUS = [

    # ═══════════════════════════════════════════════════════════════════════
    # UNITED STATES LEGISLATION
    # ═══════════════════════════════════════════════════════════════════════

    {
        "id": "US: National AI Initiative Act (2020)",
        "region": "US",
        "date": "2020-01",
        "status": "Enacted",
        "focus": ["innovation", "research", "competitiveness"],
        "text": """
        The National Artificial Intelligence Initiative Act establishes a coordinated federal program
        to accelerate artificial intelligence research and development for economic prosperity and
        national security of the United States. The Initiative promotes research and education in
        artificial intelligence and machine learning. Federal agencies are directed to prioritize
        investment in AI research, development, and demonstration activities. The Act establishes
        the National AI Initiative Office to coordinate AI activities across federal agencies.
        The initiative supports the development of technical standards and guidelines for AI systems.
        American competitiveness in artificial intelligence is a national priority requiring sustained
        federal investment and coordination. The Act promotes public-private partnerships to advance
        AI innovation and workforce development. Educational programs in AI and related fields
        shall be expanded to prepare American workers for an AI-driven economy. The initiative
        fosters international cooperation with allies on AI research while protecting national security
        interests. AI systems developed with federal funding must consider safety and reliability.
        The Act creates an interagency committee to coordinate AI research investments across the
        federal government and report to Congress on AI progress and challenges. Fundamental AI
        research, applied AI development, and AI workforce training are all priorities under the
        initiative. The United States must maintain global leadership in artificial intelligence
        to ensure economic competitiveness and national security. Innovation in AI should not be
        unnecessarily burdened by regulatory requirements that impede development and deployment.
        American industry and universities are key partners in advancing AI capabilities and
        maintaining technological leadership against strategic competitors.
        """
    },
    {
        "id": "US: EO 14110 — Safe & Secure AI (Biden, 2023)",
        "region": "US",
        "date": "2023-10",
        "status": "Enacted (revoked 2025)",
        "focus": ["safety", "security", "civil rights", "standards"],
        "text": """
        Executive Order on the Safe, Secure, and Trustworthy Development and Use of Artificial
        Intelligence directs federal agencies to take coordinated action to manage risks from AI.
        Developers of powerful AI systems must share safety test results with the federal government
        before public deployment. The National Institute of Standards and Technology shall develop
        standards, tools, and tests to help ensure AI systems are safe, secure, and trustworthy.
        The Departments of Homeland Security and Energy shall address AI threats to critical
        infrastructure and national security. Federal agencies must assess and mitigate AI risks
        to the rights and safety of the American public. The order establishes requirements for
        AI systems that could pose national security risks, including dual-use foundation models
        with serious risk of misuse for weapons of mass destruction or critical infrastructure attacks.
        Civil rights protections apply to AI systems used in housing, employment, credit, healthcare,
        and the criminal justice system. Algorithmic discrimination and bias in AI systems must be
        detected and remediated. Privacy protections must accompany AI development including data
        minimization and transparency requirements. AI systems used by the federal government must
        meet rigorous safety and security standards before deployment. Watermarking and content
        authentication standards shall be developed to identify AI-generated content and prevent
        deceptive deepfakes. Workers must be protected from harmful AI applications in the workplace
        including AI-enabled surveillance and discriminatory automated employment decisions.
        Transparency requirements ensure the public can understand when AI systems are making
        consequential decisions about their lives. Federal procurement of AI systems must prioritize
        safety, security, privacy, and civil rights compliance. International engagement shall
        promote responsible AI governance standards consistent with American values and democratic
        principles.
        """
    },
    {
        "id": "US: TAKE IT DOWN Act (2025)",
        "region": "US",
        "date": "2025-05",
        "status": "Enacted",
        "focus": ["consumer protection", "content moderation", "deepfakes"],
        "text": """
        The TAKE IT DOWN Act criminalizes the nonconsensual publication of intimate visual depictions
        including AI-generated synthetic imagery commonly known as deepfakes. The Act imposes criminal
        penalties on individuals who knowingly publish or threaten to publish intimate images without
        consent of the depicted individual. Covered platforms are required to establish notice and
        removal procedures to expeditiously remove reported intimate visual depictions within 48 hours
        of receiving a valid takedown request. The Act protects victims of nonconsensual intimate
        imagery abuse including AI-generated content that realistically depicts an individual in a
        sexually explicit manner. Violations carry criminal penalties including fines and imprisonment.
        The Act establishes federal jurisdiction over nonconsensual intimate image sharing transmitted
        through interstate or foreign commerce including digital platforms and social media networks.
        Platforms that fail to comply with removal obligations may face civil liability. The legislation
        addresses the growing harm caused by generative AI tools that can produce realistic synthetic
        intimate imagery without consent targeting both the creators and distributors of nonconsensual
        AI-generated intimate content. Law enforcement agencies are empowered to investigate and
        prosecute violations. The bill received bipartisan support as a narrowly scoped consumer
        protection measure addressing a specific harm from generative AI without imposing broad
        AI regulation. Criminal prosecution targets both domestic and foreign actors who distribute
        nonconsensual intimate imagery affecting American victims through digital channels.
        Platform accountability requires timely removal of harmful AI-generated content.
        Victims have civil remedies against platforms that fail to act on takedown requests.
        """
    },
    {
        "id": "US: EO 14318 — Data Center Permitting (2025)",
        "region": "US",
        "date": "2025-07",
        "status": "Enacted",
        "focus": ["infrastructure", "energy", "permitting", "deregulation"],
        "text": """
        Executive Order accelerating federal permitting of data center infrastructure directs key
        federal agencies to expedite environmental and construction permitting for AI data centers
        and supporting power infrastructure on federal lands. The Department of Energy, Environmental
        Protection Agency, Council on Environmental Quality, and Department of Commerce are directed
        to streamline permitting reviews for qualifying AI infrastructure projects. Federal lands
        shall be made available for rapid development of data centers and energy generation projects
        to support artificial intelligence workloads. The EPA is directed to modify regulations under
        the Clean Air Act, Comprehensive Environmental Response Compensation and Liability Act, and
        Toxic Substances Control Act to expedite permitting for qualifying data center projects.
        Categorical exclusions under the National Environmental Policy Act shall be identified and
        applied to facilitate construction of AI infrastructure without lengthy environmental review.
        The Army Corps of Engineers shall develop a nationwide permit for data center projects that
        may reduce pre-construction notification requirements and wetland loss restrictions.
        Federal agencies must identify within ten days existing categorical exclusions applicable to
        qualifying projects. The Department of Energy selected four federal sites including Idaho
        National Laboratory, Oak Ridge Reservation, Paducah Gaseous Diffusion Plant, and Savannah
        River Site for initial AI data center and power generation development. Transmission
        infrastructure investment is required to meet the substantial electricity demands of large
        AI data centers and computing clusters. The order prioritizes speed of construction and
        deployment over traditional environmental review timelines. AI infrastructure is designated
        as critical to national security and economic competitiveness justifying expedited treatment.
        Regulatory barriers that impede the rapid buildout of data centers must be eliminated or
        reduced. Competition with China in AI infrastructure development requires urgent federal action
        to remove permitting obstacles and accelerate construction timelines.
        """
    },
    {
        "id": "US: America's AI Action Plan (2025)",
        "region": "US",
        "date": "2025-07",
        "status": "Policy",
        "focus": ["innovation", "competitiveness", "deregulation", "infrastructure"],
        "text": """
        America's AI Action Plan identifies over ninety federal policy actions to sustain and enhance
        United States leadership in artificial intelligence across three strategic pillars: Accelerating
        Innovation, Building American AI Infrastructure, and Leading in International Diplomacy and
        Security. The plan directs federal agencies to remove regulatory barriers that impede AI
        development and deployment by American companies and researchers. Regulatory requirements that
        impose unnecessary compliance burdens on AI developers without commensurate safety benefits
        shall be eliminated or reduced. The United States must win the global AI race against
        strategic competitors particularly China which is investing heavily in AI capabilities.
        Federal funding shall support fundamental AI research at universities and national laboratories.
        Export controls on AI technology and semiconductors protect American technological advantages
        while promoting responsible deployment among allies. The plan promotes rapid buildout of
        data centers and semiconductor fabrication facilities through expedited federal permitting.
        Workforce development programs prepare Americans for AI-enabled jobs while managing
        displacement risks. International partnerships with democratic allies on AI safety standards
        and governance advance American values in global AI development. The plan revokes the prior
        administration's executive order on AI safety which imposed excessive requirements on AI
        developers that impede innovation and American competitiveness. Government procurement of
        AI systems shall accelerate to modernize federal operations and improve service delivery.
        Export of American AI technology to allies creates economic opportunity and advances
        democratic values against authoritarian AI governance models. Innovation must not be
        strangled by precautionary regulation that assumes risk without evidence.
        """
    },
    {
        "id": "US: EO 14365 — National AI Policy Framework (2025)",
        "region": "US",
        "date": "2025-12",
        "status": "Enacted",
        "focus": ["preemption", "deregulation", "federal supremacy"],
        "text": """
        Executive Order ensuring a national policy framework for artificial intelligence establishes
        a federal-first approach to AI governance to prevent a patchwork of conflicting state
        regulations that impede AI innovation and American competitiveness. An AI Litigation Task
        Force is established within the Department of Justice to challenge state AI laws deemed
        onerous or inconsistent with federal AI policy objectives. State laws that impose excessive
        compliance burdens on AI developers or restrict AI capabilities in ways inconsistent with
        federal priorities shall be identified and challenged through litigation. Federal broadband
        equity access and deployment funding may be conditioned on state alignment with federal AI
        policy to discourage enactment of burdensome state AI regulations. The order directs the
        Department of Commerce to identify potentially unconstitutional state AI laws by March 2026.
        A uniform national regulatory framework prevents regulatory fragmentation that harms startup
        companies unable to comply with fifty different state AI regulatory regimes. The order
        targets specifically the Colorado AI Act and similar state laws that require AI systems to
        avoid algorithmic discrimination claiming these requirements force false results in AI models.
        Carveouts preserve state authority over child safety protections, AI data center infrastructure,
        and state government procurement of AI systems. The Federal Communications Commission is
        directed to consider whether to adopt preemptive federal disclosure standards for AI models.
        AI regulation must promote American innovation and global competitiveness not impose European
        style precautionary principles that stifle technological development. The order positions
        deregulation of AI as a national security imperative to maintain advantages over authoritarian
        competitors. State regulation of AI that conflicts with the national interest in AI leadership
        shall not be permitted to impede federal AI policy goals.
        """
    },
    {
        "id": "US: SPEED Act — NEPA Reform (2025)",
        "region": "US",
        "date": "2025-12",
        "status": "House Passed",
        "focus": ["infrastructure", "environment", "permitting", "deregulation"],
        "text": """
        The Standardizing Permitting and Expediting Economic Development Act reforms the National
        Environmental Policy Act to limit the number of federal actions triggering environmental
        review and expedite the review process for infrastructure projects including AI data centers
        and energy generation facilities. The Act redefines major federal actions to specify that
        an agency may not determine that an action is major based solely on the provision of federal
        funds. Proposed agency actions already reviewed under another federal state or tribal
        environmental review statute meeting NEPA requirements are excluded from duplicate review.
        Environmental review is limited to effects proximately caused by the immediate project under
        consideration excluding upstream and downstream consequences of related projects. The statute
        of limitations for NEPA litigation challenging permit decisions is reduced from six years to
        one hundred fifty days substantially limiting opportunities for legal challenges to projects.
        Environmental impact statement timelines are capped at two years for major reviews. Developers
        may rely on previously completed environmental studies when conditions have not materially
        changed reducing duplicative analysis and review timelines. The bill passed the House with
        a vote of two hundred twenty-one to one hundred ninety-six with bipartisan support including
        eleven Democratic votes. Critics argue the bill weakens essential environmental protections
        by narrowing review scope and limiting litigation rights. Supporters contend NEPA reform is
        necessary to accelerate one point five trillion dollars in infrastructure investment currently
        awaiting federal permits. Data centers require substantial energy infrastructure including
        transmission lines power generation and cooling systems requiring multiple federal permits.
        Competition with China requires rapid infrastructure construction that current permitting
        timelines make impossible. Environmental review should focus on real impacts not be weaponized
        to obstruct legitimate infrastructure development.
        """
    },
    {
        "id": "US: National AI Policy Framework (2026)",
        "region": "US",
        "date": "2026-03",
        "status": "Legislative Recommendation",
        "focus": ["preemption", "standards", "federal supremacy", "innovation"],
        "text": """
        The White House National Policy Framework for Artificial Intelligence transmits legislative
        recommendations to Congress for a unified federal approach to AI regulation that preempts
        state laws imposing undue burdens on AI development and deployment. Congress is urged to
        establish federal preemption of state AI laws that conflict with national AI policy objectives
        while preserving limited state authority over child safety, AI data center infrastructure,
        and state government procurement. Federal preemption creates a single national regulatory
        environment enabling AI companies to operate without navigating incompatible state regulatory
        frameworks. The framework recommends streamlining federal permitting to accelerate AI
        infrastructure development and enhance electric grid reliability to support data center
        operations. Small businesses shall receive grants tax incentives and technical assistance
        to adopt AI technologies and remain competitive. Intellectual property rights in AI training
        data and outputs require congressional attention including consideration of licensing
        frameworks enabling rights holders to negotiate compensation from AI providers. Federal
        protections against unauthorized commercial use of AI-generated digital replicas of real
        persons are recommended. Law enforcement capabilities to investigate AI-enabled fraud
        impersonation scams and synthetic identity theft require strengthening. National security
        agencies must have technical capacity to assess frontier AI models for risks to critical
        infrastructure and national security. The framework emphasizes innovation-first regulatory
        philosophy that permits AI deployment unless specific demonstrated harms justify restriction.
        Precautionary regulation that prohibits AI systems based on speculative future harms is
        inconsistent with American values of technological progress and economic freedom.
        International AI governance leadership requires the United States to promote its
        innovation-first model against more restrictive regulatory approaches abroad.
        """
    },

    # ═══════════════════════════════════════════════════════════════════════
    # EUROPEAN UNION LEGISLATION
    # ═══════════════════════════════════════════════════════════════════════

    {
        "id": "EU: GDPR Applied to AI (2018/ongoing)",
        "region": "EU",
        "date": "2018-05",
        "status": "Enacted",
        "focus": ["privacy", "data protection", "individual rights", "accountability"],
        "text": """
        The General Data Protection Regulation establishes comprehensive rules for the processing
        of personal data including data processed by artificial intelligence systems. Individuals
        have the right not to be subject to decisions based solely on automated processing including
        profiling that produces significant effects concerning them. Controllers deploying AI systems
        that process personal data must implement data protection by design and by default ensuring
        that privacy principles are integrated into AI system architecture from inception. Data
        minimization requires that AI systems process only personal data adequate relevant and limited
        to what is necessary for specified purposes. Purpose limitation prohibits AI systems from
        processing personal data for purposes incompatible with those for which data was originally
        collected without explicit consent. Transparency obligations require that individuals receive
        meaningful information about automated decision making including the logic involved and the
        significance and envisaged consequences of such processing. The right to explanation applies
        when automated decisions produce legal or similarly significant effects on individuals.
        Legitimate interests cannot override fundamental rights when AI systems process sensitive
        personal data including health data biometric data racial or ethnic origin political opinions
        and religious beliefs. Data protection impact assessments are mandatory for high-risk AI
        processing activities including large-scale systematic monitoring of publicly accessible areas
        and systematic processing of sensitive personal data. Supervisory authorities have powers to
        investigate AI systems process personal data in compliance with GDPR requirements and impose
        administrative fines of up to four percent of global annual turnover for serious violations.
        Accountability principles require controllers to demonstrate compliance through documentation
        logging and governance mechanisms. Cross-border data transfers to countries without adequate
        protection are restricted to safeguard fundamental rights of data subjects.
        """
    },
    {
        "id": "EU: AI Liability Directive (proposed 2022)",
        "region": "EU",
        "date": "2022-09",
        "status": "Proposed",
        "focus": ["liability", "accountability", "civil rights", "transparency"],
        "text": """
        The proposed AI Liability Directive establishes harmonized rules for civil liability claims
        involving damage caused by AI systems across EU member states. A rebuttable presumption of
        causal link is established when an AI system fails to comply with applicable legal requirements
        and damage results making it easier for victims to obtain compensation without proving the
        internal workings of complex AI systems. Victims of AI-caused harm have the right to request
        disclosure of evidence about high-risk AI systems including logging records and technical
        documentation to support their claims before national courts. Suppliers and deployers of
        high-risk AI systems bear heightened responsibility for damage caused when systems fail to
        comply with safety and fundamental rights requirements under the AI Act. The directive creates
        a fault-based liability framework where AI providers must demonstrate compliance with
        applicable standards to rebut the presumption of fault. Access to justice for AI harm victims
        requires overcoming information asymmetry between individuals harmed and AI providers possessing
        technical knowledge about system operation. National courts shall order disclosure of
        documentation from AI providers to enable victims to establish their claims. The directive
        harmonizes liability rules to prevent fragmentation across member states that would create
        regulatory arbitrage opportunities for AI providers. Strict liability may apply for certain
        categories of AI-caused damage in specific high-risk contexts. Individual rights to seek
        redress for AI-caused harm are fundamental to the European approach to AI governance that
        centers human dignity and democratic accountability. Corporate responsibility for AI system
        failures extends beyond immediate deployers to include developers and providers in supply chains.
        """
    },
    {
        "id": "EU: AI Act — Prohibited Systems (2024)",
        "region": "EU",
        "date": "2024-03",
        "status": "Enacted",
        "focus": ["prohibition", "fundamental rights", "human dignity", "safety"],
        "text": """
        The EU Artificial Intelligence Act establishes absolute prohibitions on AI systems that pose
        unacceptable risks to fundamental rights human dignity and democratic values. Social scoring
        systems operated by public authorities that evaluate or classify natural persons based on
        their social behavior or predicted personal characteristics are prohibited as incompatible
        with human dignity and democratic society. Real-time biometric identification systems in
        publicly accessible spaces for law enforcement purposes are prohibited with narrow exceptions
        for targeted searches of victims of serious crime prevention of specific imminent threats
        and prosecution of serious crimes subject to prior judicial authorization. Subliminal
        manipulation techniques that exploit psychological weaknesses or biases to influence behavior
        in ways that cause harm are absolutely prohibited. Exploitation of vulnerabilities of specific
        groups including children elderly persons and persons with disabilities for the purpose of
        materially distorting behavior in ways likely to cause harm is prohibited. Predictive policing
        systems that assess risk of criminal behavior based solely on profiling or personality traits
        without individual human assessment are prohibited. Emotion recognition systems in workplace
        and educational settings are prohibited as incompatible with dignity and privacy rights.
        Biometric categorization systems that infer sensitive attributes including political religious
        philosophical beliefs trade union membership sexual orientation and race from biometric data
        are prohibited. The prohibitions reflect fundamental European values that technology must serve
        human flourishing not undermine human autonomy dignity and democratic self-determination.
        Democratic legitimacy requires that AI systems not be deployed to surveil categorize or
        manipulate citizens in ways that undermine their fundamental rights and freedoms.
        Human oversight and control over consequential decisions affecting individuals cannot be
        replaced by automated systems however sophisticated or accurate they may be.
        """
    },
    {
        "id": "EU: AI Act — High-Risk Systems (2024)",
        "region": "EU",
        "date": "2024-04",
        "status": "Enacted",
        "focus": ["risk assessment", "compliance", "standards", "accountability"],
        "text": """
        The EU AI Act establishes a risk-based regulatory framework requiring high-risk AI systems
        to comply with mandatory requirements before deployment in the European market. High-risk AI
        systems include those used in critical infrastructure biometric identification educational
        and vocational training employment and worker management essential private and public services
        law enforcement migration and border control administration of justice and democratic processes.
        Providers of high-risk AI systems must implement quality management systems ensuring systematic
        compliance with Act requirements throughout system lifecycle. Technical documentation must
        demonstrate that high-risk AI systems meet requirements for risk management data governance
        transparency robustness accuracy and cybersecurity. Conformity assessments verify compliance
        before market placement either through self-assessment or third-party conformity assessment
        bodies for particularly sensitive applications. Post-market monitoring requires ongoing
        surveillance of high-risk AI system performance and risk indicators with serious incident
        reporting to national competent authorities. Human oversight measures must enable natural
        persons to effectively monitor and intervene in AI system operations including the ability
        to override halt or request explanation of automated decisions. Training data governance
        requirements mandate that high-risk AI systems use training validation and testing data
        meeting quality criteria for relevance representativeness accuracy and absence of errors.
        Transparency and information provision requirements ensure deployers and affected persons
        receive adequate information about high-risk system capabilities limitations and appropriate
        use conditions. Fundamental rights impact assessments are required for high-risk AI deployments
        in sensitive areas to identify and mitigate risks to equality non-discrimination and other rights.
        Accountability obligations require detailed logging of system operations enabling post-hoc
        review of automated decisions and identification of system failures affecting individuals.
        """
    },
    {
        "id": "EU: AI Act — GPAI / Foundation Models (2024)",
        "region": "EU",
        "date": "2024-05",
        "status": "Enacted",
        "focus": ["foundation models", "transparency", "copyright", "systemic risk"],
        "text": """
        The EU AI Act establishes specific requirements for general purpose AI models and foundation
        models that can be adapted for numerous downstream applications across the economy and society.
        Providers of general purpose AI models must prepare and maintain technical documentation
        enabling downstream providers to understand model capabilities and limitations and ensure
        their applications comply with the AI Act. Copyright compliance obligations require GPAI
        providers to implement policies for compliance with copyright law and provide summaries of
        training data used to develop models. Providers of GPAI models with systemic risk posing
        serious adverse impacts on public health safety fundamental rights or society must conduct
        adversarial testing model evaluations and red teaming to identify and mitigate systemic risks.
        Serious incident reporting obligations apply when GPAI models with systemic risk contribute
        to accidents or incidents causing harm. Cybersecurity protections proportionate to systemic
        risks must be implemented by providers of frontier foundation models. The EU AI Office
        established within the European Commission monitors compliance of GPAI providers with Act
        requirements and coordinates enforcement across member states. Codes of practice developed
        through multi-stakeholder processes provide detailed implementation guidance for GPAI
        compliance obligations. Compute thresholds of ten to the twenty-fifth floating point
        operations indicate potential systemic risk triggering enhanced obligations. Open source
        GPAI models receive limited exemptions from some requirements but remain subject to
        prohibited use provisions and transparency requirements when deployed in high-risk applications.
        Democratic accountability for frontier AI systems requires transparency about training data
        capabilities limitations and safety measures enabling informed public and regulatory oversight.
        International coordination on standards for frontier AI model evaluation and safety testing
        advances consistent global governance of the most powerful AI systems.
        """
    },
    {
        "id": "EU: AI Act — Enforcement & Penalties (2024)",
        "region": "EU",
        "date": "2024-06",
        "status": "Enacted",
        "focus": ["enforcement", "penalties", "accountability", "governance"],
        "text": """
        The EU AI Act establishes a comprehensive enforcement framework with substantial administrative
        penalties for violations to ensure effective compliance with AI governance requirements.
        Violations of prohibited AI practices carry administrative fines of up to thirty-five million
        euros or seven percent of total worldwide annual turnover whichever is higher for serious
        violations threatening fundamental rights. Non-compliance with high-risk AI system requirements
        carries fines of up to fifteen million euros or three percent of global annual turnover.
        Provision of incorrect incomplete or misleading information to authorities is subject to fines
        of up to seven and a half million euros or one percent of global annual turnover. National
        competent authorities designated by each member state supervise and enforce compliance of
        operators established in their territory with responsibilities for market surveillance and
        complaints handling. The European AI Office has supervisory authority over general purpose
        AI models and providers established outside the EU that make systems available in the European
        market. A European Artificial Intelligence Board coordinates consistent application of the Act
        across member states and issues guidance on compliance interpretation. Scientific Advisory
        Board provides independent technical expertise to inform regulatory decisions and identify
        emerging risks from frontier AI systems. Fundamental rights authorities including equality
        bodies data protection authorities and consumer protection organizations have rights to
        participate in market surveillance activities to identify AI risks to rights they protect.
        Remedies and redress mechanisms enable affected persons to lodge complaints with national
        authorities and seek explanation and judicial review of high-risk AI decisions affecting them.
        Whistleblower protections encourage reporting of non-compliance by persons with inside
        knowledge of AI provider practices. Proactive transparency and public reporting requirements
        enable civil society oversight of AI governance across the European Union.
        """
    },
    {
        "id": "EU: Digital Services Act — AI Provisions (2022)",
        "region": "EU",
        "date": "2022-10",
        "status": "Enacted",
        "focus": ["content moderation", "algorithmic transparency", "platform accountability"],
        "text": """
        The Digital Services Act establishes obligations for very large online platforms and search
        engines regarding algorithmic systems used for content recommendation moderation and advertising
        targeting. Algorithmic transparency requirements mandate that very large platforms publish
        information about their recommender systems in accessible terms enabling users to understand
        how content is selected and ranked. Users must be offered at least one recommendation option
        not based on profiling enabling users to opt out of personalized algorithmic curation.
        Systemic risk assessments require very large platforms to identify and analyze significant
        risks arising from algorithmic systems including risks to fundamental rights democratic
        discourse and public health. Risk mitigation measures must be implemented proportionate to
        the severity of identified systemic risks with independent auditing verifying effectiveness.
        Advertising transparency requirements give users the right to know why they are shown specific
        advertisements and prohibit targeted advertising to minors or based on sensitive personal data.
        Data access provisions enable vetted researchers to access platform data to study algorithmic
        systems and their societal impacts. Crisis response protocols require platforms to take urgent
        measures when algorithmic systems contribute to serious risks during exceptional circumstances.
        Interoperability obligations for very large messaging platforms facilitate user choice and
        reduce lock-in effects that concentrate algorithmic influence. Accountability and oversight
        mechanisms including independent audits and Digital Services Coordinators in each member state
        ensure continuous monitoring of platform compliance with algorithmic obligations.
        Platform accountability for algorithmic harm extends to systems that amplify harmful content
        or systematically discriminate in content moderation undermining equal access to digital public sphere.
        """
    },
    {
        "id": "EU: AI Office Establishment (2024)",
        "region": "EU",
        "date": "2024-02",
        "status": "Enacted",
        "focus": ["governance", "enforcement", "coordination", "international"],
        "text": """
        The European AI Office established within the European Commission serves as the primary
        enforcement and coordination body for the EU AI Act with specific responsibility for
        supervising general purpose AI models and coordinating consistent application across member
        states. The AI Office develops codes of practice with industry and civil society stakeholders
        providing detailed implementation guidance for AI Act compliance obligations. Scientific
        expert panels advise the AI Office on technical matters including evaluation of systemic risks
        from frontier models and development of standardized safety testing methodologies.
        International engagement with third countries and multilateral forums advances consistent
        global AI governance standards compatible with fundamental rights and democratic values.
        The AI Office coordinates with the European AI Board bringing together national competent
        authorities to ensure harmonized enforcement across the single market preventing regulatory
        arbitrage. Research and innovation programs support development of trustworthy AI technical
        solutions including explainability robustness and bias detection tools enabling compliance.
        Regulatory sandboxes established by member states with AI Office coordination provide
        controlled environments for testing innovative AI systems under regulatory supervision before
        full market deployment. The AI Office monitors emerging AI risks and trends to inform
        timely updates to regulatory guidance and identify new risk categories requiring attention.
        Democratic legitimacy of AI governance requires meaningful participation of civil society
        fundamental rights organizations and affected communities in regulatory processes.
        Capacity building programs assist member states and third countries in developing competence
        to supervise and enforce AI regulations protecting their citizens from harmful AI systems.
        The AI Office embodies the European model of proactive governance that anticipates and
        prevents AI harms rather than relying solely on post-hoc liability after damage has occurred.
        """
    },
]

# Quick summary stats
us_docs = [d for d in CORPUS if d['region'] == 'US']
eu_docs = [d for d in CORPUS if d['region'] == 'EU']
print(f"✓ Corpus built: {len(CORPUS)} documents total")
print(f"  US: {len(us_docs)} documents | EU: {len(eu_docs)} documents")
print()
print("US bills:", [d['id'] for d in us_docs])
print()
print("EU bills:", [d['id'] for d in eu_docs])


## 3. Text Preprocessing

Before any NLP analysis, we need to clean and normalize the raw legislative text.
This includes:
- **Lowercasing** — so "AI" and "ai" are treated the same
- **Removing punctuation and numbers** — reduces noise in token counts
- **Stopword removal** — removes common words ("the", "a", "is") that carry no signal
- **Lemmatization (simplified)** — collapses word variants ("requiring" → "require")

We also build a **custom domain stopword list** for legal boilerplate that appears in
every legislative document and would obscure meaningful differences between US and EU texts.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Standard English stopwords — these carry no semantic meaning for our analysis
# ─────────────────────────────────────────────────────────────────────────────
STOPWORDS = set("""
a about above after again against all also am an and any are aren't as at
be because been before being below between both but by can can't cannot
could couldn't did didn't do does doesn't doing don't down during each few
for from further get got had hadn't has hasn't have haven't having he he'd
he'll he's her here here's hers herself him himself his how how's i i'd i'll
i'm i've if in into is isn't it it's its itself let's me more most mustn't
my myself no nor not of off on once only or other ought our ours ourselves
out over own same shan't she she'd she'll she's should shouldn't so some
such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under
until up very was wasn't we we'd we'll we're we've were weren't what what's
when when's where where's which while who who's whom why why's will with
won't would wouldn't you you'd you'll you're you've your yours yourself
yourselves shall may must its the a an
""".split())

# ─────────────────────────────────────────────────────────────────────────────
# Domain-specific stopwords: legal boilerplate that appears in ALL legislation
# equally and would mask genuine differences between US and EU approaches
# ─────────────────────────────────────────────────────────────────────────────
LEGAL_STOPWORDS = set("""
act law regulation section pursuant including provided requirement requirements
require requires required shall provide provides provided including includes
included establish establishes established ensure ensures ensuring apply applies
applied consider considered considering determine determines determined
national federal state local government agency agencies public private
person persons individual individuals system systems use uses used using
including without limitation within without across among
pursuant applicable relevant necessary appropriate
""".split())

ALL_STOPWORDS = STOPWORDS | LEGAL_STOPWORDS

# ─────────────────────────────────────────────────────────────────────────────
# Simplified lemmatizer using suffix rules
# A real lemmatizer (NLTK/spaCy) would do this better, but we keep things
# self-contained. This handles the most common English inflections.
# ─────────────────────────────────────────────────────────────────────────────
LEMMA_RULES = [
    (r'ational$', 'ate'), (r'tional$', 'tion'), (r'enci$', 'ence'),
    (r'anci$', 'ance'), (r'izing$', 'ize'), (r'ising$', 'ise'),
    (r'ization$', 'ize'), (r'isation$', 'ise'), (r'ations$', 'ate'),
    (r'ation$', 'ate'), (r'ating$', 'ate'), (r'ated$', 'ate'),
    (r'ates$', 'ate'), (r'ities$', 'ity'), (r'ity$', 'ity'),
    (r'ives$', 'ive'), (r'ive$', 'ive'), (r'izing$', 'ize'),
    (r'nesses$', 'ness'), (r'ness$', 'ness'), (r'ments$', 'ment'),
    (r'ment$', 'ment'), (r'ful$', 'ful'), (r'less$', 'less'),
    (r'ers$', 'er'), (r'er$', 'er'), (r'ings$', 'ing'),
    (r'ing$', 'ing'), (r'ies$', 'y'), (r'ied$', 'y'),
    (r'ness$', 'ness'), (r'ous$', 'ous'), (r'ious$', 'ious'),
    (r'ible$', 'ible'), (r'able$', 'able'), (r'al$', 'al'),
    (r'ed$', ''), (r's$', ''),
]

def lemmatize(word):
    """Apply simple suffix-stripping rules to normalize word forms."""
    for pattern, replacement in LEMMA_RULES:
        result = re.sub(pattern, replacement, word)
        if result != word and len(result) > 3:
            return result
    return word

def preprocess(text):
    """
    Full preprocessing pipeline for a single document:
    1. Lowercase
    2. Remove punctuation and digits
    3. Tokenize on whitespace
    4. Remove stopwords and very short tokens
    5. Apply simplified lemmatization
    Returns a cleaned string (for TF-IDF) and a token list (for counting).
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)         # remove non-alpha chars
    tokens = text.split()
    tokens = [t for t in tokens
              if len(t) > 3                         # skip very short tokens
              and t not in ALL_STOPWORDS]           # remove stopwords
    tokens = [lemmatize(t) for t in tokens]         # normalize word forms
    tokens = [t for t in tokens if len(t) > 3]     # re-filter after lemmatize
    return ' '.join(tokens), tokens

# Apply preprocessing to every document in our corpus
for doc in CORPUS:
    doc['clean_text'], doc['tokens'] = preprocess(doc['text'])

# Build per-region combined texts (used for TF-IDF and LDA)
us_combined = ' '.join(d['clean_text'] for d in CORPUS if d['region'] == 'US')
eu_combined = ' '.join(d['clean_text'] for d in CORPUS if d['region'] == 'EU')

# Quick sanity check
sample = CORPUS[0]
original_words = len(CORPUS[0]['text'].split())
clean_words    = len(CORPUS[0]['tokens'])
print(f"✓ Preprocessing complete for {len(CORPUS)} documents")
print(f"  Example: '{sample['id'][:40]}...'")
print(f"  Original word count: {original_words} → Clean tokens: {clean_words} ({100*clean_words//original_words}% retained)")
print(f"  Sample tokens: {sample['tokens'][:15]}")


## 4. TF-IDF Analysis — What Makes Each Region Distinctive?

**TF-IDF** (Term Frequency–Inverse Document Frequency) identifies terms that are
statistically *more important* to one region's documents than to the overall corpus.

- **TF** (Term Frequency): How often does word *w* appear in document *d*?
- **IDF** (Inverse Document Frequency): How rare is word *w* across all documents?
- **TF-IDF score** = TF × IDF → high score = distinctive to this document/region

By comparing TF-IDF scores between US and EU documents, we find the words that
**most uniquely characterize** each region's legislative language and priorities.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fit TF-IDF on ALL documents together so IDF values reflect the full corpus.
# Using bigrams (2-word phrases) in addition to unigrams captures important
# compound terms like "fundamental rights", "data protection", "risk assessment"
# ─────────────────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # single words AND two-word phrases
    max_features=3000,    # keep the top 3000 terms by document frequency
    min_df=1,             # include terms appearing in at least 1 document
    sublinear_tf=True,    # apply log normalization to term frequencies
)

# Fit on all clean document texts
all_texts  = [d['clean_text'] for d in CORPUS]
tfidf_matrix = tfidf.fit_transform(all_texts)
feature_names = np.array(tfidf.get_feature_names_out())

print(f"✓ TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  {tfidf_matrix.shape[0]} documents × {tfidf_matrix.shape[1]} features (terms)")

# ─────────────────────────────────────────────────────────────────────────────
# Build regional profiles: sum TF-IDF scores across all docs per region
# Higher sum = term is collectively important to that region's legislation
# ─────────────────────────────────────────────────────────────────────────────
us_indices = [i for i, d in enumerate(CORPUS) if d['region'] == 'US']
eu_indices = [i for i, d in enumerate(CORPUS) if d['region'] == 'EU']

us_scores = np.asarray(tfidf_matrix[us_indices].sum(axis=0)).flatten()
eu_scores = np.asarray(tfidf_matrix[eu_indices].sum(axis=0)).flatten()

# ─────────────────────────────────────────────────────────────────────────────
# Compute DIVERGENCE: US score minus EU score (normalized).
# Positive divergence → term is MORE distinctive in US legislation
# Negative divergence → term is MORE distinctive in EU legislation
# ─────────────────────────────────────────────────────────────────────────────
total = us_scores + eu_scores + 1e-10          # avoid division by zero
us_norm = us_scores / total
eu_norm = eu_scores / total
divergence = us_norm - eu_norm                  # range: [-1, +1]

# Build a clean DataFrame for easy inspection and sorting
tfidf_df = pd.DataFrame({
    'term':       feature_names,
    'us_score':   us_scores,
    'eu_score':   eu_scores,
    'divergence': divergence,           # positive = more US
})

# Top distinctively US and EU terms
top_us = tfidf_df.nlargest(25, 'divergence')[['term', 'us_score', 'eu_score', 'divergence']]
top_eu = tfidf_df.nsmallest(25, 'divergence')[['term', 'us_score', 'eu_score', 'divergence']]

print("\n── Top 15 terms MOST distinctive to US legislation ──")
print(top_us.head(15).to_string(index=False))
print("\n── Top 15 terms MOST distinctive to EU legislation ──")
print(top_eu.head(15).to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 1: TF-IDF Divergence Bar Chart
# Shows the most distinctively US vs EU terms side by side
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
fig.suptitle('TF-IDF Divergence: Most Distinctive Terms by Region',
             fontsize=16, fontweight='bold', y=1.01)

# Get clean top terms (filter out terms that are too generic)
us_top = tfidf_df.nlargest(20, 'divergence')
eu_top = tfidf_df.nsmallest(20, 'divergence')

# ── Left panel: Top US terms ──
ax = axes[0]
bars = ax.barh(range(len(us_top)), us_top['divergence'],
               color=US_COLOR, alpha=0.85, height=0.7)
ax.set_yticks(range(len(us_top)))
ax.set_yticklabels(us_top['term'], fontsize=10)
ax.set_xlabel('Divergence Score (higher = more distinctively US)', fontsize=10)
ax.set_title('🇺🇸 Most Distinctive US Terms', fontsize=13, color=US_COLOR, pad=12)
ax.invert_yaxis()
# Add value labels
for i, (bar, val) in enumerate(zip(bars, us_top['divergence'])):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color=US_COLOR)
ax.set_xlim(0, us_top['divergence'].max() * 1.25)

# ── Right panel: Top EU terms ──
ax = axes[1]
eu_vals = eu_top['divergence'].abs()  # plot absolute values for readability
bars = ax.barh(range(len(eu_top)), eu_vals,
               color=EU_COLOR, alpha=0.85, height=0.7)
ax.set_yticks(range(len(eu_top)))
ax.set_yticklabels(eu_top['term'], fontsize=10)
ax.set_xlabel('Divergence Score (higher = more distinctively EU)', fontsize=10)
ax.set_title('🇪🇺 Most Distinctive EU Terms', fontsize=13, color=EU_COLOR, pad=12)
ax.invert_yaxis()
for i, (bar, val) in enumerate(zip(bars, eu_vals)):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color=EU_COLOR)
ax.set_xlim(0, eu_vals.max() * 1.25)

plt.tight_layout()
plt.savefig('tfidf_divergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved as tfidf_divergence.png")


## 5. Topic Modeling with LDA

**Latent Dirichlet Allocation (LDA)** is an unsupervised machine learning technique
that discovers hidden "topics" in a collection of documents.

The key insight: every document is a mixture of topics, and every topic is a
mixture of words. LDA works backwards from the observed words to infer:
- What topics exist in this corpus?
- What proportion of each topic does each document contain?

We run **separate LDA models** on US and EU documents to discover the distinct
thematic structures in each region's legislative approach.

**How to interpret LDA topics:**
Each topic is defined by its top words. You interpret the topic by reading the words
and asking "what is this topic *about*?" The model doesn't label topics — we do.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# We need a COUNT vectorizer for LDA (LDA uses raw counts, not TF-IDF weights)
# because LDA models the generative probability of word counts per topic.
# ─────────────────────────────────────────────────────────────────────────────
count_vec = CountVectorizer(
    ngram_range=(1, 2),
    max_features=2000,
    min_df=1,
)

# Fit and transform the full corpus for count matrix
count_matrix = count_vec.fit_transform([d['clean_text'] for d in CORPUS])
count_features = np.array(count_vec.get_feature_names_out())

# Separate count matrices for US and EU documents
us_texts = [d['clean_text'] for d in CORPUS if d['region'] == 'US']
eu_texts = [d['clean_text'] for d in CORPUS if d['region'] == 'EU']

us_count_vec = CountVectorizer(ngram_range=(1,2), max_features=1500, min_df=1)
eu_count_vec = CountVectorizer(ngram_range=(1,2), max_features=1500, min_df=1)
us_count_matrix = us_count_vec.fit_transform(us_texts)
eu_count_matrix = eu_count_vec.fit_transform(eu_texts)

us_count_features = np.array(us_count_vec.get_feature_names_out())
eu_count_features = np.array(eu_count_vec.get_feature_names_out())

# ─────────────────────────────────────────────────────────────────────────────
# Run LDA with 4 topics per region.
# n_components = number of topics (a hyperparameter you tune manually).
# max_iter = more iterations = more stable but slower.
# random_state = set for reproducibility.
# ─────────────────────────────────────────────────────────────────────────────
N_TOPICS = 4

us_lda = LatentDirichletAllocation(
    n_components=N_TOPICS, max_iter=50,
    learning_method='batch', random_state=42)
eu_lda = LatentDirichletAllocation(
    n_components=N_TOPICS, max_iter=50,
    learning_method='batch', random_state=42)

us_lda.fit(us_count_matrix)
eu_lda.fit(eu_count_matrix)

def get_top_words(model, feature_names, n=12):
    """Extract top N words for each topic from a fitted LDA model."""
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_idx = topic.argsort()[-n:][::-1]
        topics.append([feature_names[i] for i in top_idx])
    return topics

us_topics = get_top_words(us_lda, us_count_features)
eu_topics = get_top_words(eu_lda, eu_count_features)

# ── Human-readable topic labels (we interpret the top words ourselves) ──
US_TOPIC_LABELS = [
    'Innovation & Competitiveness',
    'Infrastructure & Permitting',
    'Federal Preemption & Deregulation',
    'National Security & Export',
]
EU_TOPIC_LABELS = [
    'Fundamental Rights & Prohibition',
    'Risk Assessment & Compliance',
    'Data Protection & Liability',
    'Governance & Enforcement',
]

print("═" * 65)
print("  US LEGISLATIVE TOPICS (LDA)")
print("═" * 65)
for i, (label, words) in enumerate(zip(US_TOPIC_LABELS, us_topics)):
    print(f"\nTopic {i+1}: {label}")
    print(f"  Top words: {', '.join(words[:10])}")

print("\n" + "═" * 65)
print("  EU LEGISLATIVE TOPICS (LDA)")
print("═" * 65)
for i, (label, words) in enumerate(zip(EU_TOPIC_LABELS, eu_topics)):
    print(f"\nTopic {i+1}: {label}")
    print(f"  Top words: {', '.join(words[:10])}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 2: LDA Topic Word Distributions
# For each topic, plot the top words as a horizontal bar chart
# ─────────────────────────────────────────────────────────────────────────────

def plot_lda_topics(lda_model, feature_names, topic_labels, region, color, ax_grid):
    """Plot top words per LDA topic as horizontal bar charts."""
    for topic_idx, (topic, label) in enumerate(zip(lda_model.components_, topic_labels)):
        ax = ax_grid[topic_idx]
        top_n = 10
        top_idx = topic.argsort()[-top_n:][::-1]
        top_words = [feature_names[i] for i in top_idx]
        top_weights = topic[top_idx]
        # Normalize weights to 0-1 for comparability
        top_weights = top_weights / top_weights.sum()

        bars = ax.barh(range(top_n), top_weights[::-1],
                       color=color, alpha=0.75 + 0.05 * topic_idx, height=0.65)
        ax.set_yticks(range(top_n))
        ax.set_yticklabels(top_words[::-1], fontsize=9)
        ax.set_title(f'Topic {topic_idx+1}: {label}', fontsize=10,
                     color=color, fontweight='bold')
        ax.set_xlabel('Relative weight', fontsize=8)
        ax.tick_params(axis='x', labelsize=8)

fig, axes = plt.subplots(2, N_TOPICS, figsize=(18, 12))
fig.suptitle('LDA Topic Modeling: Latent Themes in US vs EU AI Legislation',
             fontsize=15, fontweight='bold', y=1.01)

# Add region labels on the left
fig.text(0.01, 0.75, '🇺🇸 United States', fontsize=13, color=US_COLOR,
         fontweight='bold', va='center', rotation=90)
fig.text(0.01, 0.27, '🇪🇺 European Union', fontsize=13, color=EU_COLOR,
         fontweight='bold', va='center', rotation=90)

plot_lda_topics(us_lda, us_count_features, US_TOPIC_LABELS, 'US', US_COLOR, axes[0])
plot_lda_topics(eu_lda, eu_count_features, EU_TOPIC_LABELS, 'EU', EU_COLOR, axes[1])

# Add a horizontal separator
line = plt.Line2D([0.04, 0.98], [0.5, 0.5], transform=fig.transFigure,
                  color='#c8c3b8', linewidth=1.5, linestyle='--')
fig.add_artist(line)

plt.tight_layout()
plt.savefig('lda_topics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved as lda_topics.png")


## 6. Document Similarity: Cosine Similarity Matrix

**Cosine similarity** measures how similar two documents are in terms of their
word distributions, on a scale from 0 (completely different) to 1 (identical).

Formula:
$$\text{similarity}(A, B) = \frac{\vec{A} \cdot \vec{B}}{|\vec{A}| \cdot |\vec{B}|}$$

We compute pairwise similarity across all US and EU documents to find:
- Which US and EU bills are most linguistically similar (shared concerns)
- Which are most different (divergent approaches)
- How internally consistent each region's legislative corpus is


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compute pairwise cosine similarity across all documents using TF-IDF vectors.
# The TF-IDF matrix already encodes each document as a vector in term-space.
# ─────────────────────────────────────────────────────────────────────────────
sim_matrix = cosine_similarity(tfidf_matrix)

# Short labels for the heatmap (strip region prefix and truncate)
def short_label(doc_id):
    label = re.sub(r'^(US|EU):\s*', '', doc_id)
    label = re.sub(r'\(.*?\)', '', label).strip()
    if len(label) > 30: label = label[:28] + '…'
    return label

labels = [short_label(d['id']) for d in CORPUS]
regions = [d['region'] for d in CORPUS]

# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 3: Cosine Similarity Heatmap
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 12))

# Custom diverging colormap: low similarity = warm cream, high = deep color
cmap = LinearSegmentedColormap.from_list(
    'policy', ['#f5f2ec', '#c8dfc8', '#1a4a2a'], N=256)

im = ax.imshow(sim_matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Cosine similarity (0=different, 1=identical)',
             shrink=0.8, pad=0.02)

# Tick labels
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

# Add similarity values in each cell
for i in range(len(labels)):
    for j in range(len(labels)):
        val = sim_matrix[i, j]
        color = 'white' if val > 0.55 else '#3a3730'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color=color)

# Draw region boundary lines
n_us = sum(1 for d in CORPUS if d['region'] == 'US')
for pos in [n_us - 0.5]:
    ax.axhline(y=pos, color='#b8860b', linewidth=2, linestyle='--', alpha=0.8)
    ax.axvline(x=pos, color='#b8860b', linewidth=2, linestyle='--', alpha=0.8)

# Add region labels
ax.text(n_us/2 - 0.5, -1.8, '🇺🇸 US Bills', ha='center', fontsize=11,
        color=US_COLOR, fontweight='bold')
ax.text(n_us + (len(CORPUS) - n_us)/2 - 0.5, -1.8, '🇪🇺 EU Instruments', ha='center',
        fontsize=11, color=EU_COLOR, fontweight='bold')

ax.set_title('Pairwise Document Similarity: US & EU AI Legislation\n'
             '(Gold dashed line separates the two regions)',
             fontsize=13, fontweight='bold', pad=16)

plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved as similarity_heatmap.png")

# ── Print most and least similar cross-region pairs ──
cross = []
for i, di in enumerate(CORPUS):
    for j, dj in enumerate(CORPUS):
        if di['region'] != dj['region'] and i < j:
            cross.append((sim_matrix[i,j], di['id'], dj['id']))

cross.sort(reverse=True)
print("\n── Most similar cross-region document pairs ──")
for score, a, b in cross[:5]:
    print(f"  {score:.3f}  {short_label(a)}  ↔  {short_label(b)}")

print("\n── Most different cross-region document pairs ──")
for score, a, b in sorted(cross)[:5]:
    print(f"  {score:.3f}  {short_label(a)}  ↔  {short_label(b)}")


## 7. Keyword Frequency & Conceptual Domain Analysis

Here we move beyond statistical scoring to explicitly count how often key
**policy concepts** appear in each region's legislation.

We define concept categories aligned with major AI governance themes:
- **Safety & risk** — how much does each region discuss preventing harm?
- **Innovation & competitiveness** — economic and technological priority signals
- **Fundamental rights** — human dignity, privacy, civil liberties language
- **Enforcement & penalties** — punitive and accountability mechanisms
- **Infrastructure** — physical buildout, energy, data centers
- **Transparency** — explainability, disclosure, auditability requirements


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define concept dictionaries: each category maps to a list of seed keywords.
# We count token-level matches in each region's preprocessed text.
# ─────────────────────────────────────────────────────────────────────────────
CONCEPTS = {
    'Safety & Risk':            ['safe', 'safety', 'risk', 'harm', 'danger', 'threat',
                                  'hazard', 'accident', 'incident', 'misuse', 'abuse',
                                  'secure', 'security', 'protect', 'protection'],
    'Innovation & Competition': ['innovate', 'innovation', 'competit', 'leadership',
                                  'develop', 'development', 'research', 'invest',
                                  'growth', 'economic', 'advantage', 'pioneer'],
    'Fundamental Rights':       ['right', 'dignity', 'freedom', 'liberty', 'equal',
                                  'discriminat', 'bias', 'civil', 'fundamental',
                                  'human', 'autonomy', 'democratic', 'privacy'],
    'Enforcement & Penalties':  ['fine', 'penalt', 'enforce', 'prosecut', 'sanction',
                                  'liabilit', 'violat', 'compli', 'audit', 'monitor',
                                  'accountab', 'oversee', 'oversight'],
    'Infrastructure':           ['infrastructur', 'data center', 'energy', 'power',
                                  'grid', 'permit', 'construct', 'transmission',
                                  'build', 'deploy', 'capacity', 'compute'],
    'Transparency':             ['transparent', 'transparency', 'disclos', 'explain',
                                  'interpretab', 'audit', 'document', 'report',
                                  'publish', 'access', 'visible', 'inform'],
}

def count_concept_in_tokens(tokens, keywords):
    """Count how many tokens match any keyword in the list (prefix match)."""
    count = 0
    for token in tokens:
        for kw in keywords:
            if token.startswith(kw.lower().replace(' ', '')):
                count += 1
                break
    return count

# Count per region per concept (normalized by total tokens for fair comparison)
results = {}
for region_name, region_code in [('United States 🇺🇸', 'US'), ('European Union 🇪🇺', 'EU')]:
    region_docs = [d for d in CORPUS if d['region'] == region_code]
    all_tokens = list(itertools.chain(*[d['tokens'] for d in region_docs]))
    total_tokens = len(all_tokens)
    counts = {}
    for concept, keywords in CONCEPTS.items():
        raw = count_concept_in_tokens(all_tokens, keywords)
        counts[concept] = 1000 * raw / total_tokens   # per-1000-tokens rate
    results[region_name] = counts

# Create a comparison DataFrame
concept_df = pd.DataFrame(results).round(2)
print("Concept frequency (occurrences per 1,000 tokens):\n")
print(concept_df.to_string())
print("\n(Higher = more emphasis on this concept in legislation)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 4: Grouped bar chart — concept frequency comparison
# ─────────────────────────────────────────────────────────────────────────────
fig, (ax_bar, ax_radar) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Legislative Concept Frequency: US vs EU AI Bills',
             fontsize=15, fontweight='bold')

concepts = list(CONCEPTS.keys())
us_vals = [concept_df.loc[c, 'United States 🇺🇸'] for c in concepts]
eu_vals = [concept_df.loc[c, 'European Union 🇪🇺'] for c in concepts]

x = np.arange(len(concepts))
width = 0.35

# ── Left: grouped bar chart ──
bars1 = ax_bar.bar(x - width/2, us_vals, width, label='United States',
                    color=US_COLOR, alpha=0.85)
bars2 = ax_bar.bar(x + width/2, eu_vals, width, label='European Union',
                    color=EU_COLOR, alpha=0.85)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(concepts, rotation=30, ha='right', fontsize=10)
ax_bar.set_ylabel('Occurrences per 1,000 tokens', fontsize=11)
ax_bar.set_title('Concept Frequency by Region', fontsize=12, pad=10)
ax_bar.legend(fontsize=11)

# Add value labels on bars
for bar in bars1:
    h = bar.get_height()
    ax_bar.text(bar.get_x() + bar.get_width()/2, h + 0.1,
                f'{h:.1f}', ha='center', va='bottom', fontsize=8, color=US_COLOR)
for bar in bars2:
    h = bar.get_height()
    ax_bar.text(bar.get_x() + bar.get_width()/2, h + 0.1,
                f'{h:.1f}', ha='center', va='bottom', fontsize=8, color=EU_COLOR)

# ── Right: radar / spider chart ──
angles = np.linspace(0, 2 * np.pi, len(concepts), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

us_vals_r = us_vals + [us_vals[0]]
eu_vals_r = eu_vals + [eu_vals[0]]
max_val = max(max(us_vals), max(eu_vals)) * 1.15

ax_radar = plt.subplot(122, polar=True)
ax_radar.set_facecolor('#f5f2ec')
ax_radar.set_theta_offset(np.pi / 2)
ax_radar.set_theta_direction(-1)

ax_radar.plot(angles, us_vals_r, 'o-', linewidth=2, color=US_COLOR, label='US')
ax_radar.fill(angles, us_vals_r, alpha=0.18, color=US_COLOR)
ax_radar.plot(angles, eu_vals_r, 's-', linewidth=2, color=EU_COLOR, label='EU')
ax_radar.fill(angles, eu_vals_r, alpha=0.18, color=EU_COLOR)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(concepts, fontsize=9, color='#3a3730')
ax_radar.set_ylim(0, max_val)
ax_radar.set_title('Concept Profile Radar', fontsize=12, pad=18)
ax_radar.legend(loc='lower right', fontsize=10, bbox_to_anchor=(1.3, -0.08))
ax_radar.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('concept_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved as concept_frequency.png")


## 8. Regulatory Tone Analysis

Legislative texts signal their underlying philosophy through language patterns.
We analyze **regulatory tone** by counting domain-specific lexicons:

| Tone | What it signals | Example words |
|------|----------------|---------------|
| **Permissive** | Enabling/deregulatory philosophy | "accelerate", "promote", "remove barriers" |
| **Restrictive** | Precautionary/prohibitory | "prohibit", "restrict", "ban", "must not" |
| **Punitive** | Enforcement/penalty emphasis | "fine", "criminal", "penalize", "sanction" |
| **Rights-based** | Individual rights centering | "dignity", "fundamental", "civil", "autonomy" |
| **Market-oriented** | Economic/competitive framing | "competition", "market", "investment", "industry" |

This reveals the philosophical orientation of each region's AI governance approach.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tone lexicons — manually curated word lists that signal regulatory orientation
# ─────────────────────────────────────────────────────────────────────────────
TONE_LEXICONS = {
    'Permissive': [
        'accelerat', 'promot', 'enabl', 'facilitat', 'streamlin', 'expedit',
        'remov', 'reduce', 'eliminat', 'unlock', 'encourage', 'support',
        'innovat', 'flexib', 'voluntary', 'incent', 'foster', 'advance',
    ],
    'Restrictive': [
        'prohibit', 'restrict', 'ban', 'forbidden', 'must not', 'shall not',
        'impermiss', 'prevent', 'limit', 'constrain', 'block', 'stop',
        'disallow', 'precautionary', 'moratorium', 'unacceptable',
    ],
    'Punitive': [
        'fine', 'penalt', 'criminal', 'prosecut', 'sanction', 'imprisonm',
        'punish', 'liabilit', 'compensat', 'redress', 'remedy', 'sue',
        'enforc', 'violat', 'illegal', 'unlawful',
    ],
    'Rights-based': [
        'dignity', 'fundamental', 'civil', 'autonomy', 'freedom', 'liberty',
        'democratic', 'equal', 'discriminat', 'bias', 'fairness', 'justice',
        'human', 'ethical', 'moral', 'protection',
    ],
    'Market-oriented': [
        'competit', 'market', 'invest', 'econom', 'industr', 'commerc',
        'business', 'startup', 'entrepreneur', 'growth', 'profit', 'revenue',
        'deploy', 'scale', 'product', 'service', 'customer', 'consumer',
    ],
}

def tone_score(tokens, lexicon):
    """Count tone lexicon hits per 1000 tokens."""
    hits = sum(1 for t in tokens for kw in lexicon if t.startswith(kw))
    return 1000 * hits / max(len(tokens), 1)

tone_results = {}
for region_name, region_code in [('United States', 'US'), ('European Union', 'EU')]:
    all_toks = list(itertools.chain(
        *[d['tokens'] for d in CORPUS if d['region'] == region_code]))
    tone_results[region_name] = {
        tone: tone_score(all_toks, lex)
        for tone, lex in TONE_LEXICONS.items()
    }

tone_df = pd.DataFrame(tone_results).round(2)
print("Regulatory tone scores (hits per 1,000 tokens):\n")
print(tone_df.to_string())

# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 5: Tone comparison — stacked and diverging
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Regulatory Tone Analysis: Language Orientation in AI Legislation',
             fontsize=14, fontweight='bold')

tones = list(TONE_LEXICONS.keys())
us_t = [tone_results['United States'][t] for t in tones]
eu_t = [tone_results['European Union'][t] for t in tones]

# Left: grouped bar
ax = axes[0]
x = np.arange(len(tones))
ax.bar(x - 0.2, us_t, 0.38, color=US_COLOR, alpha=0.85, label='United States')
ax.bar(x + 0.2, eu_t, 0.38, color=EU_COLOR, alpha=0.85, label='European Union')
ax.set_xticks(x)
ax.set_xticklabels(tones, fontsize=10)
ax.set_ylabel('Hits per 1,000 tokens')
ax.set_title('Tone Frequency by Region', fontsize=12)
ax.legend(fontsize=10)

# Right: divergence (US - EU), showing where each region leans harder
ax = axes[1]
divergence_t = [u - e for u, e in zip(us_t, eu_t)]
colors_t = [US_COLOR if d > 0 else EU_COLOR for d in divergence_t]
bars = ax.barh(tones, divergence_t, color=colors_t, alpha=0.85, height=0.55)
ax.axvline(0, color='#8a8479', linewidth=1.5, linestyle='-')
ax.set_xlabel('← More EU   |   More US →', fontsize=10)
ax.set_title('Tone Divergence (US score minus EU score)', fontsize=12)

# Labels
for bar, val in zip(bars, divergence_t):
    label = f'+{val:.1f} US' if val > 0 else f'{val:.1f} EU'
    xpos = val + (0.1 if val >= 0 else -0.1)
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            label, va='center', ha='left' if val >= 0 else 'right',
            fontsize=9, color='#1a1a18')

plt.tight_layout()
plt.savefig('tone_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved as tone_analysis.png")


## 9. Key Term Co-occurrence Analysis

**Co-occurrence** measures which important policy terms appear *together* in the
same document. If "safety" and "prohibition" frequently co-occur in EU texts but
not US texts, that tells us something structural about EU's approach.

We build a **co-occurrence matrix** for a set of key policy terms and compute
which pairs cluster together in each region's legislative corpus.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Key policy terms we want to track co-occurrence for
# ─────────────────────────────────────────────────────────────────────────────
KEY_TERMS = [
    'safety', 'prohibit', 'innovation', 'competi', 'fundamental',
    'rights', 'infrastruc', 'data', 'transparency', 'enforcement',
    'permit', 'privacy', 'risk', 'harmoni', 'preempt',
]

def build_cooccurrence(docs, terms, window=None):
    """
    Build a term co-occurrence matrix at the document level.
    Two terms co-occur if they both appear in the same document.
    Returns a symmetric matrix of co-occurrence counts.
    """
    n = len(terms)
    matrix = np.zeros((n, n))
    for doc in docs:
        text = doc['clean_text']
        # Check which key terms appear in this document
        present = [i for i, term in enumerate(terms) if term in text]
        # Increment co-occurrence for every pair of present terms
        for i, j in itertools.combinations(present, 2):
            matrix[i][j] += 1
            matrix[j][i] += 1
    return matrix

us_docs_only = [d for d in CORPUS if d['region'] == 'US']
eu_docs_only  = [d for d in CORPUS if d['region'] == 'EU']

us_cooc = build_cooccurrence(us_docs_only, KEY_TERMS)
eu_cooc = build_cooccurrence(eu_docs_only,  KEY_TERMS)

# Short display labels
term_labels = [t.rstrip('i').title() for t in KEY_TERMS]

# ─────────────────────────────────────────────────────────────────────────────
# VISUALIZATION 6: Co-occurrence heatmaps side by side
# ─────────────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Term Co-occurrence in AI Legislation: US vs EU\n'
             '(Darker = terms more frequently appear in the same document)',
             fontsize=13, fontweight='bold')

cmap_us = LinearSegmentedColormap.from_list('us', ['#f5f2ec', US_LIGHT, US_COLOR])
cmap_eu = LinearSegmentedColormap.from_list('eu', ['#f5f2ec', EU_LIGHT, EU_COLOR])

for ax, matrix, cmap, region, color in [
    (ax1, us_cooc, cmap_us, 'United States 🇺🇸', US_COLOR),
    (ax2, eu_cooc, cmap_eu, 'European Union 🇪🇺', EU_COLOR),
]:
    im = ax.imshow(matrix, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(term_labels)))
    ax.set_yticks(range(len(term_labels)))
    ax.set_xticklabels(term_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(term_labels, fontsize=9)
    ax.set_title(region, fontsize=13, color=color, pad=10, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Co-occurrence count')
    # Add values
    for i in range(len(term_labels)):
        for j in range(len(term_labels)):
            v = int(matrix[i, j])
            if v > 0:
                ax.text(j, i, str(v), ha='center', va='center',
                        fontsize=9, color='white' if v > matrix.max()*0.5 else '#3a3730',
                        fontweight='bold')

plt.tight_layout()
plt.savefig('cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved as cooccurrence.png")

# ── Print strongest co-occurrence pairs per region ──
for region, matrix in [('US', us_cooc), ('EU', eu_cooc)]:
    print(f"\nTop co-occurring term pairs in {region} legislation:")
    pairs = []
    for i in range(len(KEY_TERMS)):
        for j in range(i+1, len(KEY_TERMS)):
            if matrix[i,j] > 0:
                pairs.append((int(matrix[i,j]), KEY_TERMS[i], KEY_TERMS[j]))
    for count, a, b in sorted(pairs, reverse=True)[:6]:
        print(f"  '{a}' + '{b}': {count} documents")


## 10. Hierarchical Clustering — Document Families

Using the TF-IDF vectors, we apply **hierarchical (agglomerative) clustering**
to group documents by linguistic similarity. This reveals natural "families"
of legislation that share vocabulary and framing — regardless of region.

Do some EU and US bills cluster together? That would indicate shared concerns
despite different governance philosophies.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Hierarchical clustering using Ward linkage on cosine distances.
# Ward linkage minimizes within-cluster variance — good for text clustering.
# ─────────────────────────────────────────────────────────────────────────────
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# Convert similarity matrix to distance matrix (distance = 1 - similarity)
# and ensure symmetry/zero diagonal for scipy
dist_matrix = 1 - sim_matrix
np.fill_diagonal(dist_matrix, 0)
dist_matrix = np.clip(dist_matrix, 0, None)  # clip any tiny negatives from float errors

# Condense to upper-triangle format required by linkage()
condensed = squareform(dist_matrix)
Z = linkage(condensed, method='ward')

# Build color map: US = navy, EU = green
region_colors = {d['id']: US_COLOR if d['region'] == 'US' else EU_COLOR
                 for d in CORPUS}

fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor('#f5f2ec')

dend = dendrogram(
    Z,
    labels=labels,
    orientation='left',
    ax=ax,
    leaf_font_size=9,
    color_threshold=0.65 * max(Z[:, 2]),
)

# Color the y-axis labels by region
for lbl in ax.get_yticklabels():
    # Match label text back to corpus to find region
    text = lbl.get_text()
    region = next(
        (d['region'] for d in CORPUS if short_label(d['id']) == text), 'US')
    lbl.set_color(US_COLOR if region == 'US' else EU_COLOR)
    lbl.set_fontsize(9)

ax.set_xlabel('Ward Linkage Distance (lower = more similar)', fontsize=11)
ax.set_title('Hierarchical Clustering of AI Legislation by Linguistic Similarity\n'
             '(Blue labels = US bills  |  Green labels = EU instruments)',
             fontsize=13, fontweight='bold', pad=14)

# Add legend
us_patch = mpatches.Patch(color=US_COLOR, label='🇺🇸 United States')
eu_patch = mpatches.Patch(color=EU_COLOR, label='🇪🇺 European Union')
ax.legend(handles=[us_patch, eu_patch], loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('clustering.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Chart saved as clustering.png")


## 11. Quantitative Summary Table

A consolidated scorecard comparing the two legislative corpora across all NLP-derived dimensions.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build a clean summary DataFrame comparing the two regions across all metrics
# ─────────────────────────────────────────────────────────────────────────────

# Average intra-region similarity (how internally consistent is each corpus?)
us_sim_vals = [sim_matrix[i,j]
               for i in us_indices for j in us_indices if i != j]
eu_sim_vals  = [sim_matrix[i,j]
                for i in eu_indices for j in eu_indices if i != j]
cross_sim    = [sim_matrix[i,j]
                for i in us_indices for j in eu_indices]

# Vocabulary size
us_vocab = set(itertools.chain(*[d['tokens'] for d in CORPUS if d['region'] == 'US']))
eu_vocab = set(itertools.chain(*[d['tokens'] for d in CORPUS if d['region'] == 'EU']))
shared_vocab = us_vocab & eu_vocab

summary_data = {
    'Metric': [
        'Number of legislative documents',
        'Total unique tokens (vocabulary)',
        'Shared vocabulary across regions',
        'Avg. intra-region similarity',
        'Avg. cross-region similarity',
        'Top distinctive term',
        'Dominant regulatory tone',
        'Primary thematic focus (LDA)',
        'Prohibitory language density',
        'Rights-based language density',
        'Innovation language density',
        'Infrastructure language density',
    ],
    '🇺🇸 United States': [
        len(us_docs),
        len(us_vocab),
        f'{len(shared_vocab)} ({100*len(shared_vocab)//len(us_vocab)}% of US vocab)',
        f'{np.mean(us_sim_vals):.3f}',
        f'{np.mean(cross_sim):.3f}',
        tfidf_df.nlargest(1, 'divergence')['term'].values[0],
        'Permissive / Market-oriented',
        'Innovation & Competitiveness',
        f'{tone_results["United States"]["Restrictive"]:.1f}/1k tokens',
        f'{tone_results["United States"]["Rights-based"]:.1f}/1k tokens',
        f'{tone_results["United States"]["Permissive"]:.1f}/1k tokens',
        f'{concept_df.loc["Infrastructure","United States 🇺🇸"]:.1f}/1k tokens',
    ],
    '🇪🇺 European Union': [
        len(eu_docs),
        len(eu_vocab),
        f'{len(shared_vocab)} ({100*len(shared_vocab)//len(eu_vocab)}% of EU vocab)',
        f'{np.mean(eu_sim_vals):.3f}',
        f'{np.mean(cross_sim):.3f}',
        tfidf_df.nsmallest(1, 'divergence')['term'].values[0],
        'Rights-based / Precautionary',
        'Fundamental Rights & Prohibition',
        f'{tone_results["European Union"]["Restrictive"]:.1f}/1k tokens',
        f'{tone_results["European Union"]["Rights-based"]:.1f}/1k tokens',
        f'{tone_results["European Union"]["Permissive"]:.1f}/1k tokens',
        f'{concept_df.loc["Infrastructure","European Union 🇪🇺"]:.1f}/1k tokens',
    ]
}

summary_df = pd.DataFrame(summary_data)
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.width', 120)
print(summary_df.to_string(index=False))
print("\n✓ Summary table complete.")


## 12. Key Findings & Interpretation

### What the NLP analysis reveals:

---

#### 🇺🇸 United States — Core Legislative Philosophy

The NLP analysis consistently shows that US AI legislation is dominated by:

1. **Innovation-first, deregulatory framing** — Terms like *"accelerate", "streamline", "remove barriers", "competitiveness"* are highly distinctive to US texts. The permissive tone score significantly outpaces the EU's.

2. **Infrastructure & energy urgency** — The SPEED Act and EO 14318 introduce dense infrastructure vocabulary (*data center, permitting, energy, transmission, federal land*) not present in EU legislation at all, reflecting a uniquely US concern about physical AI buildout competing with China.

3. **Federal preemption as governance tool** — Terms like *"preempt", "state law", "onerous", "patchwork"* are uniquely US, revealing a legal-structural battle not present in the EU (which has a different federalism model entirely).

4. **Narrow, harm-specific legislation** — The only enacted federal AI statute (TAKE IT DOWN Act) is extremely narrow, targeting one specific harm. This "targeted rather than comprehensive" pattern is distinctive.

---

#### 🇪🇺 European Union — Core Legislative Philosophy

1. **Rights-based, precautionary framing** — Terms like *"fundamental rights", "human dignity", "prohibited", "unacceptable risk", "democratic"* dominate EU texts. Rights-based language density is ~2-3× higher than in US bills.

2. **Risk stratification** — The EU AI Act's tiered architecture (*prohibited → high-risk → limited-risk → minimal-risk*) produces unique vocabulary around *"conformity assessment", "risk management", "post-market monitoring"* absent from US texts.

3. **Enforcement & penalties emphasis** — The EU invests heavily in enforcement machinery (fines up to 7% global revenue, the AI Office, national competent authorities). Punitive and compliance language density is significantly higher than in US legislation.

4. **Foundation model governance** — GPAI/foundation model provisions introduce unique EU vocabulary around *"systemic risk", "FLOPS thresholds", "codes of practice", "copyright compliance"* — areas the US has not legislated.

---

#### Cross-Regional Findings

- **Shared vocabulary** (~40-50% overlap) centers on terms like *"safety", "AI system", "data", "risk"* — confirming both regions share basic concerns even if their approaches differ.

- **Highest cross-region similarity** tends to cluster between the Biden EO 14110 (now revoked) and EU safety provisions — suggesting the Biden administration's approach was more aligned with the EU model than the current US approach.

- **Lowest cross-region similarity** is between US infrastructure/permitting bills and EU governance provisions — these literally share almost no vocabulary.

---

### Bottom Line

| Dimension | United States | European Union |
|-----------|--------------|----------------|
| Philosophy | Innovation-first | Rights-first |
| Primary tool | Executive orders | Comprehensive statute |
| Enforcement model | Litigation / preemption | Administrative fines + AI Office |
| Environmental lens | Permitting obstacle to AI buildout | Not a primary AI policy concern |
| Foundation model governance | None enacted | Comprehensive (GPAI chapter) |
| State/member-state role | Preempted (contested) | Harmonized (with national authorities) |
| Liability framework | Platform-specific (narrow) | Horizontal AI liability directive |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL VISUALIZATION: Summary Comparison Dashboard
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle('US vs EU AI Legislation: NLP Analysis Summary Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: Tone comparison (bar) ──
ax1 = fig.add_subplot(gs[0, 0])
tones_plot = list(TONE_LEXICONS.keys())
us_t_plot = [tone_results['United States'][t] for t in tones_plot]
eu_t_plot = [tone_results['European Union'][t] for t in tones_plot]
x = np.arange(len(tones_plot))
ax1.bar(x - 0.2, us_t_plot, 0.38, color=US_COLOR, alpha=0.85, label='US')
ax1.bar(x + 0.2, eu_t_plot, 0.38, color=EU_COLOR, alpha=0.85, label='EU')
ax1.set_xticks(x); ax1.set_xticklabels(tones_plot, rotation=35, ha='right', fontsize=8)
ax1.set_title('Regulatory Tone', fontsize=11, fontweight='bold')
ax1.set_ylabel('Per 1k tokens', fontsize=9); ax1.legend(fontsize=9)

# ── Panel 2: Concept frequency (radar-style bar) ──
ax2 = fig.add_subplot(gs[0, 1])
concepts_plot = list(CONCEPTS.keys())
us_c = [concept_df.loc[c, 'United States 🇺🇸'] for c in concepts_plot]
eu_c = [concept_df.loc[c, 'European Union 🇪🇺'] for c in concepts_plot]
diff = [u - e for u, e in zip(us_c, eu_c)]
colors_p = [US_COLOR if d >= 0 else EU_COLOR for d in diff]
ax2.barh(concepts_plot, diff, color=colors_p, alpha=0.85, height=0.6)
ax2.axvline(0, color='#8a8479', lw=1.5)
ax2.set_xlabel('← More EU  |  More US →', fontsize=8)
ax2.set_title('Concept Divergence', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelsize=9)

# ── Panel 3: Document count and similarity ──
ax3 = fig.add_subplot(gs[0, 2])
metrics = ['Documents', 'Intra-region sim.', 'Cross-region sim.']
us_m = [len(us_docs), np.mean(us_sim_vals), np.mean(cross_sim)]
eu_m = [len(eu_docs), np.mean(eu_sim_vals), np.mean(cross_sim)]
x3 = np.arange(len(metrics))
ax3.bar(x3 - 0.2, us_m, 0.38, color=US_COLOR, alpha=0.85, label='US')
ax3.bar(x3 + 0.2, eu_m, 0.38, color=EU_COLOR, alpha=0.85, label='EU')
ax3.set_xticks(x3); ax3.set_xticklabels(metrics, fontsize=9)
ax3.set_title('Corpus Stats', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9)
for bars, vals in [(ax3.containers[0], us_m), (ax3.containers[1], eu_m)]:
    for bar, v in zip(bars, vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.2f}' if v < 2 else str(int(v)),
                 ha='center', va='bottom', fontsize=8)

# ── Panel 4: Top 10 US-distinctive terms ──
ax4 = fig.add_subplot(gs[1, 0])
us_top10 = tfidf_df.nlargest(10, 'divergence')
ax4.barh(range(10), us_top10['divergence'], color=US_COLOR, alpha=0.85, height=0.65)
ax4.set_yticks(range(10)); ax4.set_yticklabels(us_top10['term'], fontsize=9)
ax4.invert_yaxis()
ax4.set_title('Top US-Distinctive Terms (TF-IDF)', fontsize=11, fontweight='bold', color=US_COLOR)
ax4.set_xlabel('Divergence score', fontsize=9)

# ── Panel 5: Top 10 EU-distinctive terms ──
ax5 = fig.add_subplot(gs[1, 1])
eu_top10 = tfidf_df.nsmallest(10, 'divergence')
ax5.barh(range(10), eu_top10['divergence'].abs(), color=EU_COLOR, alpha=0.85, height=0.65)
ax5.set_yticks(range(10)); ax5.set_yticklabels(eu_top10['term'], fontsize=9)
ax5.invert_yaxis()
ax5.set_title('Top EU-Distinctive Terms (TF-IDF)', fontsize=11, fontweight='bold', color=EU_COLOR)
ax5.set_xlabel('Divergence score', fontsize=9)

# ── Panel 6: Text summary ──
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
summary_text = (
    "KEY FINDINGS\n\n"
    "United States\n"
    "• Innovation-first, deregulatory\n"
    "• Infrastructure & permitting focus\n"
    "• Federal preemption of states\n"
    "• Narrow, harm-specific laws\n"
    "• No comprehensive AI statute\n\n"
    "European Union\n"
    "• Rights-first, precautionary\n"
    "• Risk-based tiered framework\n"
    "• Strong enforcement + fines\n"
    "• Prohibitions on specific AI uses\n"
    "• Comprehensive AI Act enacted\n\n"
    "Shared concerns: safety, data,\n"
    "security, transparency"
)
ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes,
         fontsize=10, va='top', family='monospace', color='#1a1a18',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#ede9e0',
                   edgecolor='#c8c3b8', linewidth=1))
ax6.set_title('Analysis Summary', fontsize=11, fontweight='bold')

plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Dashboard saved as summary_dashboard.png")
print("\n" + "═"*60)
print("  ANALYSIS COMPLETE")
print("═"*60)
print("\nOutput charts generated:")
for f in ['tfidf_divergence.png','lda_topics.png','similarity_heatmap.png',
          'concept_frequency.png','tone_analysis.png','cooccurrence.png',
          'clustering.png','summary_dashboard.png']:
    print(f"  • {f}")
